In [1]:
import sys
sys.path.append("../")

from dotenv import load_dotenv
from sqlalchemy import create_engine, text, Table, MetaData, inspect
from sqlalchemy.orm import Session
from sqlalchemy.dialects.postgresql import insert as pg_insert
import pandas as pd
import os

from src.db_gestion import DataBaseGestion
from src.ddragon import pipeline_champion

In [2]:
db = DataBaseGestion()

load_dotenv()
database_url = os.getenv("DATABASE_URL")
engine = create_engine(database_url)
metadata = MetaData()
metadata.reflect(bind=engine)  # Charge toutes les tables depuis la DB

In [3]:
table_champion, table_champion_version, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up, table_runes, table_patch = pipeline_champion()

Lastest version : 16.5.1


In [4]:
def pipeline_insert(table_name : str, df_insert : pd.DataFrame):
    with Session(engine) as session:
        nb = db.insert_rows(session, table_name, df_insert)
        session.commit()
        print(f"{nb} lignes insérées dans {table_name}")

def pipeline_update(table_name : str, df_maj : pd.DataFrame):
    with Session(engine) as session:
        nb = db.update_rows(session, table_name, df_maj)
        session.commit()
        print(f"{nb} lignes mises à jour dans {table_name}")

def pipeline_delete(table_name : str, filters : dict):
    with Session(engine) as session:
        nb = db.delete_rows(session, table_name, filters=filters)
        session.commit()
        print(f"{nb} lignes supprimées dans {table_name}")

def pipeline_drop_table(tables_name : list[str], cascade : bool):
    if cascade == False:
        db.drop_tables(engine, tables_name) # Sans CASCADE (échoue s'il y a des FK dépendantes)
    else:
        db.drop_tables(engine, tables_name, cascade=cascade) # Avec CASCADE (supprime aussi les tables liées par FK)

In [5]:
def main():
    pipeline_insert(table_name = "patch", df_insert = table_patch)
    pipeline_insert(table_name = "champion", df_insert = table_champion)
    pipeline_insert(table_name = "champion_version", df_insert = table_champion_version)
    pipeline_insert(table_name = "champion_info", df_insert = table_champ_info)
    pipeline_insert(table_name = "champion_passive", df_insert = table_champ_passive)
    pipeline_insert(table_name = "champion_spells", df_insert = table_champ_spells)
    pipeline_insert(table_name = "champion_stats", df_insert = table_champ_stats)
    pipeline_insert(table_name = "champion_stats_up", df_insert = table_champ_stats_up)

In [6]:
main()

0 lignes insérées dans patch
0 lignes insérées dans champion
0 lignes insérées dans champion_version
0 lignes insérées dans champion_info
0 lignes insérées dans champion_passive
0 lignes insérées dans champion_spells
0 lignes insérées dans champion_stats
0 lignes insérées dans champion_stats_up
